# Apply Classifier

In [1]:
from torch.nn.functional import softmax
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import AutoTokenizer, DataCollatorWithPadding

In [2]:
processed_data_dir = Path('processed_data/tous')
file_name = processed_data_dir / 'metadata.tsv' # _annotated_distilbert
metadata = pd.read_csv(file_name,sep='\t')
metadata.fillna('', inplace=True)
print(len(metadata))

124495


In [ ]:
metadata.columns

In [ ]:
clause_type = 'modification' # 'opt-out' | 'arbitration' | 'class waiver' | 'anti-scraping' | 'modification'
model = AutoModelForSequenceClassification.from_pretrained(f'models/{clause_type}_best_model_ft_ds')
tokenizer = AutoTokenizer.from_pretrained(f'models/{clause_type}_best_model_ft_ds')

In [ ]:
#model.to('mps')

In [ ]:
metadata.head()

In [ ]:
metadata['sentence_processed_2'] = metadata.sentence.str.lower().str.strip()
mapping = metadata.groupby("sentence_processed_2").apply(lambda x: x.index.tolist()).to_dict()
len(mapping)

In [ ]:
sent2class = {s: softmax(model(**tokenizer(s, return_tensors='pt', truncation=True)).logits.detach(), dim=1)   
                         for s in tqdm(mapping.keys())}

In [ ]:
tqdm.pandas()
metadata[f'logits_{clause_type}']  = metadata.sentence_processed_2.progress_apply(lambda x: sent2class[x])
# metadata[f'logits_{clause_type}'] = metadata.progress_apply(lambda x: 
#                     softmax(model(**tokenizer(x.sentence, return_tensors='pt', truncation=True).to('mps')).logits.detach(), dim=1), 
#                           axis=1)



In [ ]:
metadata[f'prob_1_{clause_type}'] = metadata[f'logits_{clause_type}'].apply(lambda x: x[0][1].item())
metadata[clause_type] = .0
metadata.loc[metadata[f'prob_1_{clause_type}'] > .9, clause_type] = 1

In [ ]:
file_name

In [ ]:
metadata.to_csv(processed_data_dir /f'metadata_annotated_distilbert.tsv', sep='\t', index=False)

In [ ]:
metadata[clause_type].value_counts()

In [ ]:
metadata.columns

In [ ]:
metadata['year_int'] = metadata.year.apply(lambda x: int(str(x)[:4]))

In [ ]:
import seaborn as sns

data = metadata.groupby(['platform','year_int'])[clause_type].sum().astype(bool).astype(int).unstack().fillna(-1)



#.loc['bumble'].plot(kind='bar')

In [ ]:
import pandas as _pd

def replace_minus_ones_with_prev(X, axis=1, inplace=False):
    """
    Replace -1 entries in a matrix/array with the nearest preceding 0 or 1 along the given axis.
    If there is no preceding non -1 value, the -1 is left unchanged.

    Parameters:
    - X: array-like (numpy array, list of lists, or pandas DataFrame)
    - axis: 1 to replace along rows (left-to-right), 0 to replace along columns (top-to-bottom)
    - inplace: if True and X is a numpy array or DataFrame, modify it in place; otherwise return a new array

    Returns:
    - numpy.ndarray or pandas.DataFrame with replacements applied (unless inplace=True modifies input)
    """


    is_df = _pd is not None and isinstance(X, _pd.DataFrame)
    if is_df:
        arr = X.values
    else:
        arr = X if isinstance(X, (np.ndarray,)) else np.array(X)

    if not inplace:
        arr = arr.copy()

    if axis not in (0, 1):
        raise ValueError("axis must be 0 or 1")

    # iterate over the chosen axis and carry forward the last seen non -1 value
    if axis == 1:
        # rows
        for r in range(arr.shape[0]):
            last = None
            for c in range(arr.shape[1]):
                val = arr[r, c]
                if val != -1:
                    last = val
                elif last is not None:
                    arr[r, c] = last
    else:
        # columns
        for c in range(arr.shape[1]):
            last = None
            for r in range(arr.shape[0]):
                val = arr[r, c]
                if val != -1:
                    last = val
                elif last is not None:
                    arr[r, c] = last

    if is_df:
        if inplace:
            X.iloc[:, :] = arr
            return X
        else:
            return _pd.DataFrame(arr, index=X.index, columns=X.columns)
    else:
        return arr

sns.set(rc={'figure.figsize':(6.7,10.27)})
sns.heatmap(replace_minus_ones_with_prev(data),cbar=False)

In [ ]:
metadata.columns

In [ ]:
#metadata.drop(columns=['logits_arbitration', 'logits_anti-scraping'], inplace=True)

In [ ]:
metadata.columns

In [ ]:
#metadata.sort_values('prob_1', ascending=False).head(10)

In [ ]:
df_deduplicated = metadata.drop_duplicates(subset=['sentence'])
df_deduplicated['annotated'] = df_deduplicated.sentence.isin(df_annotations.text)
int_labels = [((0.95,1.0),'confident_positive'),( (0.80,.95), 'sure_positive'),((0.60,.80), 'leaning_positive'),
                   ((0.50,.60), 'borderline_positive'),((0.40,.50), 'borderline_negative'),
                   ((0.20,.40), 'leaning_negative'),((0.05,.20), 'sure_negative'),((0.0,.05), 'confident_negative')]
for interval, label in int_labels:


    df_deduplicated.loc[df_deduplicated.prob_1.between(*interval),'category']  = label



In [ ]:
pd.concat([df_deduplicated[df_deduplicated.category == label].sample(10)
    for _ , label in int_labels], axis=0)[['sentence','category']].to_csv(f'annotations/inference/{clause_type}_automatic_annotations_by_category.csv')


In [ ]:
metadata[metadata.prob_1 > .5].to_csv(f'annotations/inference/{clause_type}_inference.csv')

## Fin